Nếu chạy trên Google Colab, upload file này lên Google Colab và chạy dòng dưới đây. Sau khi xong, chạy lệnh "pwd" để xem đã vào đúng directory Transformer-NLP-UET hay chưa.

Nếu chạy trên máy, hãy bỏ qua nó.

In [1]:
!git clone -b complete https://github.com/BobInVietnam/Transformer-NLP-UET.git
%cd /content/Transformer-NLP-UET

Cloning into 'Transformer-NLP-UET'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 121 (delta 42), reused 99 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 18.03 MiB | 21.64 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/Transformer-NLP-UET


In [2]:
import pandas as pd
from transformer.transformer import Transformer

import os
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader

from data.vocab import Vocabulary
from data.dataset import SummaryDataset
from data.dataloader import collate_fn

# Load dữ liệu từ dataset

In [3]:
train_df = pd.read_parquet("dataset/train-00000-of-00001.parquet")
valid_df = pd.read_parquet("dataset/valid-00000-of-00001.parquet")
print(train_df)


                                                 article  \
0      Gần 20 sự kiện được tổ chức trên toàn thành ph...   
1      Được thành lập năm 1897 tại Đức, Kempinski Hot...   
2      Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...   
3      Với những tín đồ Phật giáo, bức tượng Phật ngọ...   
4      Số liệu của Tổng cục Thống kê công bố sáng 29/...   
...                                                  ...   
10770  Tình hình an ninh tại một số bang miền bắc Mya...   
10771  Tòa Hình sự Thượng thẩm quận Suwon hôm nay tuy...   
10772  Caolan Gormley, 26 tuổi, chủ công ty vận tải H...   
10773  Trong bối cảnh tình hình Myanmar đang xảy ra g...   
10774  Bộ Công an cho biết tính đến ngày 6/12, Cục Qu...   

                                                 summary  
0      Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...  
1      Kempinski Hotels là một thương hiệu nổi tiếng ...  
2      Bài viết giới thiệu các hoạt động vui chơi giả...  
3      Bức tượng Phật ngọc hòa bình thế giớ

In [4]:
print(train_df.columns)

Index(['article', 'summary'], dtype='object')


# Thông số model
Block code dưới đây nhằm load thông số model qua *model_config*, bao gồm các thông số về **số lượng từ vựng**, **độ lớn vector embedding**, **số lớp của model**, **số attention head**, **độ lớn lớp feed-forward** 

*checkpoint_path* chỉ vào 1 file .pt (file chứa các thông số của model bao gồm optimizer để tiếp tục train). File này sẽ được xuất ra sau mỗi epoch ở phần training bên dưới.

Sau khi chỉnh sửa  thông số, chạy tất cả các block bên dưới bắt đầu từ block này.

In [5]:
start_epoch = 0
state_dict_to_load = None
optimizer_state_to_load = None
model_config = {
    "src_vocab_size": 30000,
    "tgt_vocab_size": 30000,
    "d_model": 512,
    "num_layers": 6,
    "num_heads": 8,
    "d_ff": 2048
}

BATCH_SIZE = 16

checkpoint_path = ""

In [ ]:
def load_checkpoint(checkpoint_path: str):
    if os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from: {checkpoint_path}...")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Extract historical tracking metadata
        start_epoch = checkpoint["epoch"]
        model_config = checkpoint["config"]
        state_dict_to_load = checkpoint["model_state_dict"]
        optimizer_state_to_load = checkpoint["optimizer_state_dict"]
        print(f"Checkpoint loaded successfully. Resuming from Epoch {start_epoch + 1}")
    else:
        print("No file found yet")

load_checkpoint(checkpoint_path)

No file found yet


Phần code load dữ liệu vào lớp vocab nhằm indexing.

In [7]:
train_input_vocab = Vocabulary(max_vocab=model_config["src_vocab_size"])
train_input_vocab.build_vocab(train_df["article"])

train_output_vocab = Vocabulary(max_vocab=model_config["tgt_vocab_size"])
train_output_vocab.build_vocab(train_df["summary"])

In [8]:
print(len(train_output_vocab.stoi))

30000


In [9]:
from data.dataloader import get_dataloader
train_dataloader = get_dataloader(dataframe=train_df, src_vocab=train_input_vocab, tgt_vocab=train_output_vocab, batch_size=BATCH_SIZE)
val_dataloader = get_dataloader(dataframe=valid_df, src_vocab=train_input_vocab, tgt_vocab=train_output_vocab, batch_size=BATCH_SIZE)

# Training the base Transformer

In [10]:

def train_epoch(model, dataloader, optimizer, criterion, device):
    """
    Trains the Transformer model for a single epoch.

    Args:
        model: The top-level unified Transformer module.
        dataloader: The train_dataloader fetching padded batch dictionaries.
        optimizer: The optimizer (e.g., torch.optim.Adam).
        criterion: The loss function (e.g., nn.CrossEntropyLoss).
        device: The compute hardware target ('cuda' or 'cpu').

    Returns:
        float: The average loss for this training epoch.
    """
    # 1. Put the model in training mode (activates Dropout layers)
    model.train()

    total_loss = 0.0

    # 2. Wrap your dataloader with tqdm to build the terminal progress bar
    # 'desc' adds a prefix label; 'leave=True' keeps the progress bar visible after completion
    progress_bar = tqdm(dataloader, desc="Training Epoch", leave=True)

    for batch_idx, batch in enumerate(progress_bar):
        # 3. Extract the named tensors from your custom collate_fn output dictionary
        input_ids = batch["input_ids"].to(device)                   # Source tokens
        attention_mask = batch["attention_mask"].unsqueeze(1).unsqueeze(2).to(device)           # Encoder padding mask
        decoder_input_ids = batch["decoder_input_ids"].to(device)   # Right-shifted target tokens
        labels = batch["labels"].to(device)                         # Expected output values

        # 4. Standard PyTorch optimization reset step
        optimizer.zero_grad()

        logits = model(src=input_ids, tgt=decoder_input_ids, src_mask=attention_mask)

        vocab_size = logits.size(-1)
        flat_logits = logits.view(-1, vocab_size)
        flat_labels = labels.view(-1)

        loss = criterion(flat_logits, flat_labels)

        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        total_loss += current_loss
        running_avg_loss = total_loss / (batch_idx + 1)

        progress_bar.set_postfix({
            "batch_loss": f"{current_loss:.4f}",
            "avg_loss": f"{running_avg_loss:.4f}"
        })

    return total_loss / len(dataloader)

In [11]:
def validate_epoch(model, dataloader, criterion, device):
    """
    Evaluates the model on a validation dataset using Teacher Forcing
    to calculate the validation loss.
    """
    # 1. Put the model in evaluation mode (turns off Dropout)
    model.eval()

    total_loss = 0.0

    # 2. Wrap dataloader with tqdm progress tracking
    progress_bar = tqdm(dataloader, desc="Validation Epoch", leave=True)

    # 3. Disable gradient calculation to save memory and compute speed
    with torch.no_grad():
        for batch_idx, batch in enumerate(progress_bar):
            # Extract variables from your custom collate_fn dictionary
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].unsqueeze(1).unsqueeze(2).to(device)
            decoder_input_ids = batch["decoder_input_ids"].to(device)
            labels = batch["labels"].to(device)

            # Forward pass through the Transformer
            logits = model(src=input_ids, tgt=decoder_input_ids, src_mask=attention_mask)

            # Flatten shapes for CrossEntropy Loss
            vocab_size = logits.size(-1)
            flat_logits = logits.view(-1, vocab_size)
            flat_labels = labels.view(-1)

            # Calculate loss (ignores padding index 0 if configured in criterion)
            loss = criterion(flat_logits, flat_labels)

            total_loss += loss.item()
            running_avg_loss = total_loss / (batch_idx + 1)

            # Update the tqdm progress bar string on-the-fly
            progress_bar.set_postfix({
                "val_batch_loss": f"{loss.item():.4f}",
                "val_avg_loss": f"{running_avg_loss:.4f}"
            })

    return total_loss / len(dataloader)

In [ ]:
# 1. Choose your device hardware target
def train(num_epochs: int):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing pipeline on device: {device}")

    transformer_model = Transformer(
        src_vocab_size=model_config["src_vocab_size"],
        tgt_vocab_size=model_config["tgt_vocab_size"],
        d_model=model_config["d_model"],
        num_heads=model_config["num_heads"],
        num_layers=model_config["num_layers"],
        d_ff=model_config["d_ff"]
    )

    if state_dict_to_load is not None:
        transformer_model.load_state_dict(state_dict_to_load)

    transformer_model.to(device)

    # 3. Setup Loss criterion
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    # 4. Setup Optimizer (Matching the original paper setup parameters)
    optimizer = torch.optim.Adam(transformer_model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

    if optimizer_state_to_load is not None:
        optimizer.load_state_dict(optimizer_state_to_load)

    # 5. Execute your complete epoch execution loop
    for epoch in range(start_epoch, num_epochs):
        print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")

        # Run a training pass across the dataset via your dataloader
        epoch_loss = train_epoch(
            model=transformer_model,
            dataloader=train_dataloader,
            optimizer=optimizer,
            criterion=criterion,
            device=device
        )
        if (epoch + 1) % 2 == 0:
            val_loss = validate_epoch(
                model=transformer_model,
                dataloader=val_dataloader,
                criterion=criterion,
                device=device)

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": transformer_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": epoch_loss,
            "config": model_config  # <-- Injected directly here!
        }

        torch.save(checkpoint, "transformer_checkpoint.pt")
        print("Self-contained checkpoint saved cleanly.")

        print(f"Epoch {epoch + 1} Summary -> Overall Average Loss: {epoch_loss:.4f}")

In [13]:
train(10)

Executing pipeline on device: cuda

--- Epoch 1/10 ---


Training Epoch: 100%|██████████| 674/674 [10:11<00:00,  1.10it/s, batch_loss=5.8751, avg_loss=6.6951]


Self-contained checkpoint saved cleanly.
Epoch 1 Summary -> Overall Average Loss: 6.6951

--- Epoch 2/10 ---


Training Epoch:   2%|▏         | 14/674 [00:13<10:51,  1.01it/s, batch_loss=5.5108, avg_loss=5.7672]


KeyboardInterrupt: 